In [46]:
import os
import json
from pathlib import Path
from uuid import uuid4
from datetime import datetime
from typing import Any, Optional, Literal

from dotenv import load_dotenv

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_ollama import ChatOllama
from langchain_openrouter import ChatOpenRouter
from langchain_groq import ChatGroq



from langgraph.checkpoint.memory import MemorySaver
from pydantic import BaseModel, Field

from dotenv import load_dotenv
from difflib import unified_diff


In [2]:
load_dotenv()

True

In [3]:
OPENROUTER_MODEL = os.getenv(
    "OPENROUTER_MODEL",
    "minimax/minimax-m2.5:free",
)

In [4]:
OPENROUTER_MODEL

'openrouter/free'

In [49]:
import re
from pathlib import Path
from uuid import uuid4

# -------------------------------------------------------------------
# Simple file formats
# -------------------------------------------------------------------

TIME_MD_TEMPLATE = """# Time

Format:
- when | chapter | summary

## Events
"""

ELEMENTS_MD_TEMPLATE = """# Elements

Format:
- kind | display_name | uuid | aliases | identification_keys

Notes:
- aliases are comma-separated
- identification_keys are semicolon-separated short recognition cues

## Entries
"""

RICH_ELEMENT_FILE_TEMPLATE = """# {name}

## Identification
- UUID: {uuid}
- Type: {kind}
- Canonical name: {name}
- Aliases: {aliases}
- Identification keys: {identification_keys}

## Core Understanding
{core_understanding}

## Stable Profile
- TBD

## Interpretation
- TBD

## Knowledge / Beliefs / Uncertainties
- TBD

## Element-Centered Chronology
### TBD
- TBD

## Open Threads
- TBD
"""

# -------------------------------------------------------------------
# Boilerplate helpers
# -------------------------------------------------------------------

def read_text(path: str | Path) -> str:
    path = Path(path)
    return path.read_text(encoding="utf-8") if path.exists() else ""

def write_text(path: str | Path, content: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")

def ensure_bootstrap_file(path: str | Path, default_content: str) -> None:
    path = Path(path)
    if not path.exists() or not path.read_text(encoding="utf-8").strip():
        write_text(path, default_content)

def generate_element_uuid(prefix: str = "elt") -> str:
    return f"{prefix}_{uuid4().hex[:12]}"

def normalize_lookup(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# -------------------------------------------------------------------
# Markdown parsers + match helpers
# -------------------------------------------------------------------

def parse_elements_md(text: str) -> dict:
    """
    Parses:
    - kind | display_name | uuid | aliases | identification_keys

    aliases -> comma-separated
    identification_keys -> semicolon-separated
    """
    lookup = {}
    records = {}

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line.startswith("- "):
            continue

        parts = [p.strip() for p in line[2:].split("|")]
        if len(parts) < 5:
            continue

        kind, display_name, element_uuid, aliases_raw, identification_keys_raw = parts[:5]
        aliases = [a.strip() for a in aliases_raw.split(",") if a.strip()]
        identification_keys = [k.strip() for k in identification_keys_raw.split(";") if k.strip()]

        record = {
            "kind": kind,
            "display_name": display_name,
            "uuid": element_uuid,
            "aliases": aliases,
            "identification_keys": identification_keys,
        }

        records[element_uuid] = record

        candidate_lookups = [display_name] + aliases + identification_keys
        for value in candidate_lookups:
            norm = normalize_lookup(value)
            if norm and norm not in lookup:
                lookup[norm] = record

    return {
        "lookup": lookup,
        "records": records,
    }

def resolve_existing_element(existing_index: dict, candidate: dict) -> dict | None:
    """
    Matching order:
    1. display_name
    2. aliases
    3. identification_keys
    """
    lookup = existing_index.get("lookup", {})

    candidate_values = (
        [candidate["display_name"]]
        + candidate.get("aliases", [])
        + candidate.get("identification_keys", [])
    )

    for value in candidate_values:
        norm = normalize_lookup(value)
        if norm and norm in lookup:
            return lookup[norm]

    return None

def build_elements_entry(
    kind: str,
    display_name: str,
    element_uuid: str,
    aliases: list[str] | None = None,
    identification_keys: list[str] | None = None,
) -> str:
    aliases = aliases or []
    identification_keys = identification_keys or []
    aliases_str = ", ".join(aliases)
    identification_keys_str = "; ".join(identification_keys)
    return f"- {kind} | {display_name} | {element_uuid} | {aliases_str} | {identification_keys_str}"

def build_element_file_content(
    name: str,
    kind: str,
    element_uuid: str,
    snapshot: str,
    aliases: list[str] | None = None,
    identification_keys: list[str] | None = None,
) -> str:
    aliases = aliases or []
    identification_keys = identification_keys or []

    return RICH_ELEMENT_FILE_TEMPLATE.format(
        name=name,
        uuid=element_uuid,
        kind=kind,
        aliases=", ".join(aliases) if aliases else "-",
        identification_keys="; ".join(identification_keys) if identification_keys else "-",
        core_understanding=snapshot.strip() if snapshot.strip() else "TBD",
    )

# -------------------------------------------------------------------
# Example diff knowledge base (mock only)
# This is the part you will later replace with the real agent.
# -------------------------------------------------------------------

EXAMPLE_ELEMENT_CATALOG = {
    "Mira": {
        "kind": "person",
        "aliases": [],
        "identification_keys": [
            "central viewpoint character",
            "carries silver key",
            "linked to Arun",
            "linked to Elias",
            "tied to Saint Alder Chapel",
        ],
        "snapshot": "Central viewpoint character connected to the chapel, the greenhouse, Elias, her mother, and the silver key.",
        "update_hint": "Mira receives a letter, carries the silver key again, learns Elias was not her cousin, and leaves the chapel with letters, a watch, and a ledger page.",
    },
    "Arun": {
        "kind": "person",
        "aliases": [],
        "identification_keys": [
            "close companion of Mira",
            "works in greenhouse",
            "present at chapel investigation",
        ],
        "snapshot": "Mira's close companion in the greenhouse and chapel investigation.",
        "update_hint": "Arun is present in both chapters, shares family knowledge about Elias, and accompanies Mira to the chapel and river path.",
    },
    "Elias": {
        "kind": "person",
        "aliases": ["Elias Vale"],
        "identification_keys": [
            "missing figure from ten years earlier",
            "linked to river disappearance",
            "connected to toll house ledger",
            "mother's brother",
        ],
        "snapshot": "A missing figure from ten years earlier whose identity and family relationship are central to the mystery.",
        "update_hint": "Elias is referenced as missing by the river, associated with the toll house ledger, and revealed to be Mira's mother's brother rather than Mira's cousin.",
    },
    "Mother": {
        "kind": "person",
        "aliases": ["Mira's mother"],
        "identification_keys": [
            "sends letter at night",
            "linked to hidden truth about Elias",
        ],
        "snapshot": "Mira's mother sends the midnight letter and is tied to the hidden truth about Elias.",
        "update_hint": "She sends the letter and is implicated in the false family story about Elias.",
    },
    "Nani": {
        "kind": "person",
        "aliases": [],
        "identification_keys": [
            "painted blue pot",
            "elder who told different truth about Elias",
        ],
        "snapshot": "An elder figure connected to the blue pot and to earlier truths about Elias.",
        "update_hint": "Nani painted the blue pot and was the only one who said Elias neither ran nor drowned.",
    },
    "Sister Celine": {
        "kind": "person",
        "aliases": ["Celine"],
        "identification_keys": [
            "religious figure at chapel",
            "vanished from parish records",
            "knows about silver key",
        ],
        "snapshot": "A vanished religious figure who reappears inside Saint Alder Chapel and knows about the silver key.",
        "update_hint": "Sister Celine appears in the chapel, warns Mira about the key, and disappears with the bundle.",
    },
    "Saint Alder Chapel": {
        "kind": "place",
        "aliases": ["chapel"],
        "identification_keys": [
            "religious site",
            "linked to silver key",
            "linked to Sister Celine",
            "near Hush Road",
        ],
        "snapshot": "A central religious site tied to Mira's past, the silver key, and the current mystery.",
        "update_hint": "The chapel is entered at 7:15 a.m.; footprints, a bundle, a watch, a ledger page, and Sister Celine are all encountered there.",
    },
    "greenhouse": {
        "kind": "place",
        "aliases": [],
        "identification_keys": [
            "early morning meeting place",
            "linked to Mira and Arun",
            "contains blue pot",
        ],
        "snapshot": "The early morning meeting place where Mira and Arun begin the chapter.",
        "update_hint": "Mira and Arun meet in the greenhouse before sunrise, then leave it at 6:05 a.m.",
    },
    "Hush Road": {
        "kind": "place",
        "aliases": [],
        "identification_keys": [
            "road near greenhouse",
            "road near chapel",
            "logging trucks pass here",
        ],
        "snapshot": "A road near the greenhouse and chapel, used as a spatial anchor in both chapters.",
        "update_hint": "Logging trucks pass along Hush Road, and the road is again mentioned after the chapel scene.",
    },
    "toll house": {
        "kind": "place",
        "aliases": [],
        "identification_keys": [
            "linked to ledger",
            "connected to Elias warning",
        ],
        "snapshot": "A place connected to the ledger and Elias's earlier warning.",
        "update_hint": "The toll house ledger is referenced in Mira's memory and in the torn ledger page found in the chapel.",
    },
    "river": {
        "kind": "place",
        "aliases": ["river path"],
        "identification_keys": [
            "linked to Elias disappearance",
            "exit route after chapel",
        ],
        "snapshot": "A place linked to Elias's disappearance and the characters' exit route.",
        "update_hint": "Elias is described as missing by the river, and Mira and Arun leave toward the river path.",
    },
    "silver key": {
        "kind": "item",
        "aliases": ["key"],
        "identification_keys": [
            "found under chapel floorboards",
            "carried by Mira",
            "recognized by Sister Celine",
        ],
        "snapshot": "A recurring object tied to the chapel floorboards and a hidden mechanism or truth.",
        "update_hint": "Mira carries the silver key again; Sister Celine directly reacts to it.",
    },
    "blue pot": {
        "kind": "item",
        "aliases": [],
        "identification_keys": [
            "painted with white cranes",
            "associated with Nani",
            "in greenhouse",
        ],
        "snapshot": "A painted pot associated with Nani and the greenhouse.",
        "update_hint": "The blue pot appears in the greenhouse and is left on the sill when Mira and Arun depart.",
    },
    "watch": {
        "kind": "item",
        "aliases": ["cracked watch"],
        "identification_keys": [
            "found in chapel bundle",
            "stopped at 2:12 a.m.",
            "later ticking again",
        ],
        "snapshot": "A watch found in the chapel bundle, initially stopped at 2:12 a.m., then ticking again.",
        "update_hint": "The cracked watch appears in the bundle, is linked to 2:12 a.m., and remains after Sister Celine vanishes.",
    },
    "letters": {
        "kind": "item",
        "aliases": ["packet of letters"],
        "identification_keys": [
            "late-night letter from mother",
            "older letters dated July 3 1988",
        ],
        "snapshot": "Letters tied to Mira's mother and to older concealed events.",
        "update_hint": "A letter arrives at 11:40 p.m., and later a packet of letters dated July 3, 1988 is recovered.",
    },
    "ledger page": {
        "kind": "item",
        "aliases": ["toll house ledger page"],
        "identification_keys": [
            "torn page from toll house ledger",
            "reveals Elias relationship truth",
        ],
        "snapshot": "A torn ledger page that reveals the truth about Elias's family relationship.",
        "update_hint": "The torn ledger page identifies Elias Vale as Mira's mother's brother.",
    },
    "river stones": {
        "kind": "item",
        "aliases": ["stones"],
        "identification_keys": [
            "three stones in chapel bundle",
            "left at altar",
        ],
        "snapshot": "Three river stones left as part of the bundle at the altar.",
        "update_hint": "Three river stones are found in the cloth bundle in the chapel.",
    },
}

def mock_extract_identified_elements(diff_text: str) -> list[dict]:
    """
    Stand-in for the main agent's element identification.
    This is intentionally simple and grounded to the example diff.
    """
    found = []
    lowered = diff_text.lower()

    for display_name, meta in EXAMPLE_ELEMENT_CATALOG.items():
        names_to_check = (
            [display_name]
            + meta.get("aliases", [])
            + meta.get("identification_keys", [])
        )
        if any(name.lower() in lowered for name in names_to_check[: len([display_name] + meta.get("aliases", []))]):
            found.append({
                "display_name": display_name,
                "kind": meta["kind"],
                "aliases": meta.get("aliases", []),
                "identification_keys": meta.get("identification_keys", []),
                "snapshot": meta["snapshot"],
                "update_hint": meta["update_hint"],
            })

    found.sort(key=lambda x: (x["kind"], x["display_name"].lower()))
    return found

def mock_extract_time_entries(diff_text: str) -> list[dict]:
    """
    Stand-in for the main agent's time reasoning.
    Keeps the output compact and useful.
    """
    entries = []

    if "Late June, 1998" in diff_text:
        entries.append({
            "when": "1998-06 late month | 05:40-06:05",
            "chapter": "Chapter 7",
            "summary": "Mira meets Arun in the greenhouse, receives context from her mother's late-night letter, recalls Elias and the toll house ledger, and leaves for Saint Alder Chapel carrying the silver key.",
        })

    if "June 28, 1998" in diff_text:
        entries.append({
            "when": "1998-06-28 | 07:15-07:32",
            "chapter": "Chapter 8",
            "summary": "Mira and Arun enter Saint Alder Chapel, find the bundle, watch, and torn ledger page, encounter Sister Celine, recover letters, and leave toward the river path.",
        })

    if "ten years earlier" in diff_text:
        entries.append({
            "when": "approx. 1988-06",
            "chapter": "Backstory reference in Chapter 7",
            "summary": "Mira recalls Elias standing on the chapel path with wet cuffs and a bloodied knuckle, telling her not to reveal what he saw in the toll house ledger.",
        })

    if "July 3, 1988" in diff_text:
        entries.append({
            "when": "1988-07-03",
            "chapter": "Backstory artifact in Chapter 8",
            "summary": "A packet of letters dated July 3, 1988 is found beneath the vestry drawer lining.",
        })

    if "11:40 p.m." in diff_text:
        entries.append({
            "when": "1998-06 previous night | 23:40",
            "chapter": "Pre-scene reference in Chapter 7",
            "summary": "Mira receives her mother's letter late at night before the greenhouse meeting.",
        })

    if "2:12 a.m." in diff_text:
        entries.append({
            "when": "1998-06-28 | 02:12",
            "chapter": "Artifact timestamp in Chapter 8",
            "summary": "The cracked watch found in the chapel bundle is initially stopped at 2:12 a.m.",
        })

    return entries

# -------------------------------------------------------------------
# Main mock planner
# -------------------------------------------------------------------

def build_mock_main_agent_plan(diff_text: str, current_elements_md: str, current_time_md: str) -> dict:
    """
    This is the function you will later replace with the real main agent call.
    For now it returns the STRUCTURE you want from the agent.
    """
    existing_index = parse_elements_md(current_elements_md)
    identified = mock_extract_identified_elements(diff_text)
    time_entries = mock_extract_time_entries(diff_text)

    resolved_elements = []
    new_elements = []
    new_element_files = {}

    for item in identified:
        existing = resolve_existing_element(existing_index, item)

        if existing:
            resolved = {
                "display_name": existing["display_name"],
                "kind": existing["kind"],
                "uuid": existing["uuid"],
                "aliases": existing["aliases"],
                "identification_keys": existing.get("identification_keys", []),
                "is_new": False,
                "snapshot": item["snapshot"],
                "update_hint": item["update_hint"],
            }
        else:
            element_uuid = generate_element_uuid()
            resolved = {
                "display_name": item["display_name"],
                "kind": item["kind"],
                "uuid": element_uuid,
                "aliases": item["aliases"],
                "identification_keys": item["identification_keys"],
                "is_new": True,
                "snapshot": item["snapshot"],
                "update_hint": item["update_hint"],
            }
            new_elements.append(resolved)
            new_element_files[element_uuid] = build_element_file_content(
                name=item["display_name"],
                kind=item["kind"],
                element_uuid=element_uuid,
                snapshot=item["snapshot"],
                aliases=item["aliases"],
                identification_keys=item["identification_keys"],
            )

        resolved_elements.append(resolved)

    elements_md_lines = [
        build_elements_entry(
            kind=e["kind"],
            display_name=e["display_name"],
            element_uuid=e["uuid"],
            aliases=e["aliases"],
            identification_keys=e["identification_keys"],
        )
        for e in new_elements
    ]

    time_md_lines = [
        f"- {entry['when']} | {entry['chapter']} | {entry['summary']}"
        for entry in time_entries
    ]

    element_update_tasks = [
        {
            "uuid": e["uuid"],
            "display_name": e["display_name"],
            "kind": e["kind"],
            "aliases": e["aliases"],
            "identification_keys": e["identification_keys"],
            "update_instruction": e["update_hint"],
        }
        for e in resolved_elements
    ]

    return {
        "identified_elements": resolved_elements,
        "new_elements": new_elements,
        "time_entries": time_entries,
        "proposed_changes": {
            "append_to_time_md": "\n".join(time_md_lines).strip(),
            "append_to_elements_md": "\n".join(elements_md_lines).strip(),
            "create_element_files": new_element_files,
        },
        "element_update_tasks": element_update_tasks,
    }

# -------------------------------------------------------------------
# Approval / apply functions
# -------------------------------------------------------------------

def preview_main_plan(plan: dict) -> None:
    print("\n=== IDENTIFIED ELEMENTS ===")
    for e in plan["identified_elements"]:
        status = "NEW" if e["is_new"] else "EXISTING"
        keys = "; ".join(e.get("identification_keys", [])) if e.get("identification_keys") else "-"
        print(f"- {e['display_name']} ({e['kind']}) [{status}] -> {e['uuid']}")
        print(f"  aliases: {', '.join(e.get('aliases', [])) or '-'}")
        print(f"  identification_keys: {keys}")

    print("\n=== PROPOSED time.md APPEND ===")
    print(plan["proposed_changes"]["append_to_time_md"] or "[no change]")

    print("\n=== PROPOSED elements.md APPEND ===")
    print(plan["proposed_changes"]["append_to_elements_md"] or "[no change]")

    print("\n=== NEW ELEMENT FILES ===")
    for k in plan["proposed_changes"]["create_element_files"].keys():
        print(f"- {k}.md")

def apply_main_plan(
    plan: dict,
    time_md_path: str | Path,
    elements_md_path: str | Path,
    elements_dir: str | Path,
    approved: bool = False,
) -> None:
    """
    Only applies changes if approved=True.
    This is your 'user permission' guard.
    """
    if not approved:
        raise PermissionError("Plan not applied. Set approved=True only after user confirmation.")

    time_md_path = Path(time_md_path)
    elements_md_path = Path(elements_md_path)
    elements_dir = Path(elements_dir)

    ensure_bootstrap_file(time_md_path, TIME_MD_TEMPLATE)
    ensure_bootstrap_file(elements_md_path, ELEMENTS_MD_TEMPLATE)
    elements_dir.mkdir(parents=True, exist_ok=True)

    time_append = plan["proposed_changes"]["append_to_time_md"].strip()
    if time_append:
        current = read_text(time_md_path).rstrip()
        write_text(time_md_path, current + "\n" + time_append + "\n")

    elements_append = plan["proposed_changes"]["append_to_elements_md"].strip()
    if elements_append:
        current = read_text(elements_md_path).rstrip()
        write_text(elements_md_path, current + "\n" + elements_append + "\n")

    for element_uuid, file_content in plan["proposed_changes"]["create_element_files"].items():
        element_path = elements_dir / f"{element_uuid}.md"
        if not element_path.exists():
            write_text(element_path, file_content)

# -------------------------------------------------------------------
# Optional: single convenience function for your notebook
# -------------------------------------------------------------------

def run_mock_main_step(
    diff_path: str | Path,
    time_md_path: str | Path,
    elements_md_path: str | Path,
) -> dict:
    diff_text = read_text(diff_path)
    current_time_md = read_text(time_md_path)
    current_elements_md = read_text(elements_md_path)

    return build_mock_main_agent_plan(
        diff_text=diff_text,
        current_elements_md=current_elements_md,
        current_time_md=current_time_md,
    )

In [17]:
ensure_bootstrap_file("story/time.md", TIME_MD_TEMPLATE)
ensure_bootstrap_file("story/elements.md", ELEMENTS_MD_TEMPLATE)

plan = run_mock_main_step(
    diff_path="example_story_change.diff",
    time_md_path="story/time.md",
    elements_md_path="story/elements.md",
)

preview_main_plan(plan)


=== IDENTIFIED ELEMENTS ===
- blue pot (item) [NEW] -> elt_b337f6d76794
  aliases: -
  identification_keys: painted with white cranes; associated with Nani; in greenhouse
- ledger page (item) [NEW] -> elt_3ba8879ce615
  aliases: toll house ledger page
  identification_keys: torn page from toll house ledger; reveals Elias relationship truth
- letters (item) [NEW] -> elt_0fc555a1511f
  aliases: packet of letters
  identification_keys: late-night letter from mother; older letters dated July 3 1988
- river stones (item) [NEW] -> elt_c21b9a321c83
  aliases: stones
  identification_keys: three stones in chapel bundle; left at altar
- silver key (item) [NEW] -> elt_6d45e39bdd38
  aliases: key
  identification_keys: found under chapel floorboards; carried by Mira; recognized by Sister Celine
- watch (item) [NEW] -> elt_87c889cbe906
  aliases: cracked watch
  identification_keys: found in chapel bundle; stopped at 2:12 a.m.; later ticking again
- Arun (person) [NEW] -> elt_9ad971181122
  alias

In [18]:
apply_main_plan(
    plan,
    time_md_path="story/time.md",
    elements_md_path="story/elements.md",
    elements_dir="story/elements",
    approved=True,
)

## Lanchain impl

In [51]:
# ------------------------------------------------------------
# Structured output for time.md proposal
# ------------------------------------------------------------

class TimeEvent(BaseModel):
    when: str = Field(description="Canonical or approximate time expression")
    chapter: str = Field(description="Chapter or source section")
    summary: str = Field(description="Compact reader-friendly event summary")

class TimeUpdateProposal(BaseModel):
    changed: bool = Field(description="Whether time.md needs a new append block")
    rationale: str = Field(description="Why these entries should be added")
    events: list[TimeEvent] = Field(default_factory=list)
    append_block: str = Field(description="Exact markdown lines to append to time.md")
    approval_message: str = Field(description="Short message shown to the human reviewer")


In [52]:
# ------------------------------------------------------------
# System prompt
# ------------------------------------------------------------

TIME_AGENT_SYSTEM_PROMPT = """
You are the Time Ledger Agent.

Your only job is to read:
1. a manuscript git diff
2. the current contents of time.md

Then propose updates for time.md.

You are not a story critic.
You do not judge whether the writing is strange, wrong, implausible, or incomplete.
You only extract a clean, reader-friendly time ledger from what the writer has added or changed.

Rules:
- Prefer compact, high-level entries.
- Include explicit timestamps when present.
- Include approximate backstory timing when clearly referenced.
- If a chapter spans a period, summarize that period compactly.
- Do not invent dates or times not grounded in the diff.
- Do not modify files directly.
- Return only a structured proposal for human approval.

Formatting rule for append_block:
- Each line must follow:
  - when | chapter | summary
"""

In [53]:


time_model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=8000,
    max_retries=2,
    timeout=120,
).with_structured_output(TimeUpdateProposal)

In [54]:
def propose_time_md_update(
    diff_path: str | Path,
    time_md_path: str | Path,
    reviewer_feedback: Optional[str] = None,
) -> TimeUpdateProposal:
    diff_text = read_text(diff_path)
    current_time_md = read_text(time_md_path)

    feedback_block = reviewer_feedback.strip() if reviewer_feedback else "None"

    user_prompt = f"""
Current time.md:

<time_md>
{current_time_md}
</time_md>

Incoming manuscript diff:

<diff>
{diff_text}
</diff>

Reviewer feedback for this revision:
{feedback_block}

Task:
Propose whether time.md should be updated.
If yes, return the exact markdown append block and the structured event list.
If no, set changed=false and leave append_block empty.
"""

    return time_model.invoke([
        {"role": "system", "content": TIME_AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ])


In [55]:
# ------------------------------------------------------------
# Preview / apply helpers
# ------------------------------------------------------------

def preview_time_proposal(proposal: TimeUpdateProposal) -> None:
    print("\n=== TIME UPDATE PROPOSAL ===")
    print("changed:", proposal.changed)
    print("\nrationale:")
    print(proposal.rationale)

    print("\nproposed append_block:")
    print(proposal.append_block or "[no change]")

    print("\nevents:")
    if not proposal.events:
        print("[none]")
    else:
        for event in proposal.events:
            print(f"- {event.when} | {event.chapter} | {event.summary}")

    print("\napproval_message:")
    print(proposal.approval_message)

def apply_time_md_update(
    proposal: TimeUpdateProposal,
    time_md_path: str | Path,
    approved: bool = False,
) -> None:
    if not approved:
        raise PermissionError("Not applied. Pass approved=True only after review.")

    if not proposal.changed or not proposal.append_block.strip():
        print("No time.md changes to apply.")
        return

    time_md_path = Path(time_md_path)
    ensure_bootstrap_file(time_md_path, TIME_MD_TEMPLATE)

    current = read_text(time_md_path).rstrip()
    new_content = current + "\n" + proposal.append_block.strip() + "\n"
    write_text(time_md_path, new_content)
    print(f"Applied update to {time_md_path}")

In [56]:
proposal = propose_time_md_update(
    diff_path="example_story_change.diff",
    time_md_path="story/time.md",
)
preview_time_proposal(proposal)


=== TIME UPDATE PROPOSAL ===
changed: True

rationale:
The manuscript diff adds precise timestamps and new details (letter arrival time, bundle contents, ledger revelation, and Sister Celine encounter) that are not yet captured in time.md. Adding these entries provides a compact, high‑level ledger of the newly described events.

proposed append_block:
- 1998-06-27 | 23:40 | Chapter 7 | Mira receives her mother’s letter sealed with chapel wax at 11:40 p.m., containing a vague reference to “the one who went missing by the river”.
- 1998-06-28 | 07:15-07:32 | Chapter 8 | Mira and Arun discover a cloth bundle with three river stones, a cracked watch stopped at 2:12 a.m., and a torn toll‑house ledger page naming Elias Vale as their mother’s brother; they are confronted by Sister Celine, who warns them about the silver key.
- 1998-06-28 | 07:32 | Chapter 8 | The pair leave the chapel via the side door, taking the letters, watch, and ledger page, and head toward the river path.

events:
- 19

In [33]:
# reject with a comment
proposal = propose_time_md_update(
    diff_path="example_story_change.diff",
    time_md_path="story/time.md",
    reviewer_feedback="Too granular. Merge the late-night letter into the Chapter 7 entry and keep only one backstory line.",
)
preview_time_proposal(proposal)

In [58]:
#approve
apply_time_md_update(
    proposal,
    time_md_path="story/time.md",
    approved=True,
)

Applied update to story/time.md


In [59]:
# ------------------------------------------------------------
# Structured output
# ------------------------------------------------------------

class ElementDecision(BaseModel):
    display_name: str = Field(
        description="Canonical display name for the element. Prefer the simplest stable canonical name, not a temporary phrase from the diff."
    )
    kind: Literal["person", "place", "item", "animal", "concept", "group", "other"]

    aliases: list[str] = Field(
        default_factory=list,
        description="Alternative names or surface forms explicitly supported by the diff or the existing index."
    )

    identification_keys: list[str] = Field(
        default_factory=list,
        description="3-6 short recognition cues that help distinguish this element from others. These must not simply repeat the display name."
    )

    snapshot: str = Field(
        description="2-3 sentence high-level reader-friendly summary of what this element is in the story as understood from the diff and current index."
    )

    update_instruction: str = Field(
        description="A concrete instruction for the later uuid.md updater describing what this diff implies should be added or revised for this element."
    )

    evidence_from_diff: list[str] = Field(
        default_factory=list,
        description="2-6 short exact phrases or names from the diff that justify identifying this element."
    )

    matched_existing_display_name: Optional[str] = Field(
        default=None,
        description="Exact canonical display name from elements.md if this matches an existing element."
    )

    matched_existing_uuid: Optional[str] = Field(
        default=None,
        description="UUID from elements.md if this matches an existing element."
    )

    is_new: bool = Field(
        description="True only if this element does not match any existing indexed element."
    )


class ElementsProposal(BaseModel):
    diff_summary: str = Field(
        description="2-5 sentence summary of what the diff adds or changes."
    )

    rationale: str = Field(
        description="Short explanation of how existing-vs-new decisions were made."
    )

    identified_elements: list[ElementDecision] = Field(
        default_factory=list,
        description="All relevant durable story elements in the diff."
    )

    approval_message: str = Field(
        description="Short human-facing summary of what will be added to elements.md if approved."
    )

elements_model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=8000,
    max_retries=2,
    timeout=120,
).with_structured_output(ElementsProposal)

In [60]:
# ------------------------------------------------------------
# System prompt
# ------------------------------------------------------------

ELEMENTS_AGENT_SYSTEM_PROMPT = """
You are the Elements Index Agent.

Your job is to read:
1. the manuscript git diff
2. the current elements.md index

Then produce a proposal that identifies all durable story-relevant elements mentioned or materially affected by the diff.

A durable story-relevant element may be:
- a person
- a place
- an item
- an animal
- a group
- a concept
- another concrete recurring element that matters to the story world

Do not index trivial background noise.
Do not index every grass blade, every generic wall, or every incidental unnamed object.
Do index elements that matter to understanding the story world, relationships, events, places, or recurring objects.

Your output must be reader-friendly, auditable, and useful for downstream uuid.md updates.

Core responsibilities:
- identify all relevant durable elements in the diff
- match them to existing elements.md entries when the evidence is strong
- propose new elements only when no strong existing match exists
- produce strong identification metadata for each element
- produce a useful update instruction for the later per-element updater

Important rules:

1. You are not a critic.
Do not judge the story as weird, wrong, implausible, or incomplete.
Your job is to make the world model legible, not to correct the writer.

2. Prefer stable canonical naming.
Use the simplest stable canonical name.
Do not overfit the canonical name to a temporary descriptive phrase.
For example, prefer:
- "watch" over "cracked watch" when the cracked watch is clearly the same item
- "ledger page" over "torn ledger page" when the tear is a temporary property
- "Elias" over "Elias Vale" if the diff indicates that is the core recurring identity and the surname is just a fuller mention
Only use the longer name as canonical if the longer form is necessary for disambiguation.

3. identification_keys must be real discriminators.
They must not simply restate the display name.
Bad examples:
- Mira
- chapel
- silver key
Good examples:
- carries the silver key
- linked to Saint Alder Chapel
- missing figure from ten years earlier
- found in the chapel bundle
Use short phrases, not full paragraphs.

4. update_instruction must be concrete.
Do not write lazy generic instructions like:
- Add new person entry for Mira
- Add new item entry
Instead describe what this diff implies should be captured later in the uuid.md.
Good example:
- Record that Mira receives her mother's letter, resumes carrying the silver key, recalls Elias and the toll house ledger, and leaves the chapel with the recovered letters, watch, and ledger page.

5. snapshot must be useful.
Write 2-3 sentences that help a reader understand the element's current role in the world model.
Do not make it vague or generic.

6. evidence_from_diff must be grounded.
Include short exact phrases or names from the diff that justify the identification.
Do not invent evidence.

7. Existing-vs-new matching:
Prefer matching an existing element when there is strong evidence from:
- exact name match
- alias match
- clear conceptual match
Do not create a new element just because the diff uses a slightly different surface phrase.
Resolve to the same underlying thing when appropriate.

8. Coverage:
Return all relevant durable elements in the diff, not just the first few obvious ones.

9. Do not create UUIDs.
If an element matches an existing entry, copy the matched existing display name and UUID.
If it is new, leave matched_existing_uuid empty and set is_new=true.

Output quality bar:
- canonical names should be stable
- aliases should be meaningful
- identification_keys should distinguish the element
- snapshots should be reader-friendly
- update_instruction should be downstream-useful
- evidence_from_diff should justify the decision
"""

In [61]:
# ------------------------------------------------------------
# Proposal builder
# ------------------------------------------------------------

def propose_elements_update(
    diff_path: str | Path,
    elements_md_path: str | Path,
    reviewer_feedback: Optional[str] = None,
) -> dict:
    diff_text = read_text(diff_path)
    current_elements_md = read_text(elements_md_path)
    existing_index = parse_elements_md(current_elements_md)

    feedback_block = reviewer_feedback.strip() if reviewer_feedback else "None"

    user_prompt = f"""
Current elements.md:

<elements_md>
{current_elements_md}
</elements_md>

Incoming diff:

<diff>
{diff_text}
</diff>

Reviewer feedback:
{feedback_block}

Task:
Identify the elements involved in this change and decide which are existing vs new.
If an element clearly matches an existing indexed element, include the exact matched_existing_display_name and matched_existing_uuid.
If it is new, set is_new=true and leave matched_existing_uuid empty.
"""

    proposal = elements_model.invoke([
        {"role": "system", "content": ELEMENTS_AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ])

    # Finalize deterministically in Python
    finalized_elements = []
    new_elements = []
    new_element_files = {}

    for item in proposal.identified_elements:
        item_dict = item.model_dump()

        existing = None
        if item_dict.get("matched_existing_uuid"):
            existing = existing_index["records"].get(item_dict["matched_existing_uuid"])

        if existing is None:
            existing = resolve_existing_element(existing_index, item_dict)

        if existing:
            resolved = {
                "display_name": existing["display_name"],
                "kind": existing["kind"],
                "uuid": existing["uuid"],
                "aliases": existing["aliases"],
                "identification_keys": existing.get("identification_keys", []),
                "is_new": False,
                "snapshot": item_dict["snapshot"],
                "update_instruction": item_dict["update_instruction"],
            }
        else:
            new_uuid = generate_element_uuid()
            resolved = {
                "display_name": item_dict["display_name"],
                "kind": item_dict["kind"],
                "uuid": new_uuid,
                "aliases": item_dict.get("aliases", []),
                "identification_keys": item_dict.get("identification_keys", []),
                "is_new": True,
                "snapshot": item_dict["snapshot"],
                "update_instruction": item_dict["update_instruction"],
            }
            new_elements.append(resolved)
            new_element_files[new_uuid] = build_element_file_content(
                name=resolved["display_name"],
                kind=resolved["kind"],
                element_uuid=resolved["uuid"],
                snapshot=resolved["snapshot"],
                aliases=resolved["aliases"],
                identification_keys=resolved["identification_keys"],
            )

        finalized_elements.append(resolved)

    append_lines = [
        build_elements_entry(
            kind=e["kind"],
            display_name=e["display_name"],
            element_uuid=e["uuid"],
            aliases=e["aliases"],
            identification_keys=e["identification_keys"],
        )
        for e in new_elements
    ]

    return {
        "rationale": proposal.rationale,
        "approval_message": proposal.approval_message,
        "identified_elements": finalized_elements,
        "new_elements": new_elements,
        "proposed_changes": {
            "append_to_elements_md": "\n".join(append_lines).strip(),
            "create_element_files": new_element_files,
        },
        "final_element_update_targets": [
            {
                "uuid": e["uuid"],
                "display_name": e["display_name"],
                "kind": e["kind"],
                "update_instruction": e["update_instruction"],
            }
            for e in finalized_elements
        ],
    }

In [62]:
# ------------------------------------------------------------
# Preview / apply
# ------------------------------------------------------------

def preview_elements_proposal(plan: dict) -> None:
    print("\n=== ELEMENTS PROPOSAL ===")
    print(plan["rationale"])

    print("\n=== IDENTIFIED ELEMENTS ===")
    for e in plan["identified_elements"]:
        status = "NEW" if e["is_new"] else "EXISTING"
        print(f"- {e['display_name']} ({e['kind']}) [{status}] -> {e['uuid']}")
        print(f"  aliases: {', '.join(e['aliases']) or '-'}")
        print(f"  identification_keys: {'; '.join(e['identification_keys']) or '-'}")
        print(f"  update_instruction: {e['update_instruction']}")

    print("\n=== PROPOSED elements.md APPEND ===")
    print(plan["proposed_changes"]["append_to_elements_md"] or "[no change]")

    print("\n=== NEW ELEMENT FILES ===")
    for uuid in plan["proposed_changes"]["create_element_files"]:
        print(f"- {uuid}.md")

    print("\n=== FINAL UPDATE TARGETS ===")
    for t in plan["final_element_update_targets"]:
        print(f"- {t['display_name']} -> {t['uuid']}")

    print("\n=== APPROVAL MESSAGE ===")
    print(plan["approval_message"])

def apply_elements_proposal(
    plan: dict,
    elements_md_path: str | Path,
    elements_dir: str | Path,
    approved: bool = False,
) -> None:
    if not approved:
        raise PermissionError("Not applied. Pass approved=True only after review.")

    elements_md_path = Path(elements_md_path)
    elements_dir = Path(elements_dir)

    ensure_bootstrap_file(elements_md_path, ELEMENTS_MD_TEMPLATE)
    elements_dir.mkdir(parents=True, exist_ok=True)

    append_block = plan["proposed_changes"]["append_to_elements_md"].strip()
    if append_block:
        current = read_text(elements_md_path).rstrip()
        write_text(elements_md_path, current + "\n" + append_block + "\n")

    for element_uuid, file_content in plan["proposed_changes"]["create_element_files"].items():
        element_path = elements_dir / f"{element_uuid}.md"
        if not element_path.exists():
            write_text(element_path, file_content)

    print(f"Applied elements proposal to {elements_md_path}")

def apply_run_manifest(
    plan: dict,
    runs_dir: str | Path,
    approved: bool = False,
) -> Path:
    if not approved:
        raise PermissionError("Not applied. Pass approved=True only after review.")

    runs_dir = Path(runs_dir)
    runs_dir.mkdir(parents=True, exist_ok=True)

    now = datetime.now()
    run_id = now.strftime("%Y-%m-%dT%H-%M-%S")
    run_path = runs_dir / f"{run_id}.md"

    lines = [
        "# Run Manifest",
        f"- run_id: {run_id}",
        f"- created_at: {now.isoformat()}",
        "",
        "## Element Update Targets",
    ]

    for element in plan["identified_elements"]:
        lines.extend([
            f"- uuid: {element['uuid']}",
            f"  display_name: {element['display_name']}",
            f"  kind: {element['kind']}",
            f"  file: story/elements/{element['uuid']}.md",
            f"  update_instruction: {element['update_instruction']}",
            "",
        ])

    write_text(run_path, "\n".join(lines).rstrip() + "\n")
    print(f"Applied run manifest to {run_path}")
    return run_path

In [63]:
elements_plan = propose_elements_update(
    diff_path="example_story_change.diff",
    elements_md_path="story/elements.md",
)
preview_elements_proposal(elements_plan)


=== ELEMENTS PROPOSAL ===
The existing elements index is empty, so every durable entity mentioned in the diff is treated as new. Each entry captures a distinct person, place, or object that recurs or holds narrative significance, with clear evidence, identification cues, and actionable update instructions for later UUID assignment.

=== IDENTIFIED ELEMENTS ===
- Mira (person) [NEW] -> elt_6ad4c9d565e1
  aliases: -
  identification_keys: protagonist; carries silver key; searches chapel; recalls Elias
  update_instruction: Record Mira as the main protagonist who carries the silver key, receives her mother's letter, recalls Elias, and explores the chapel with Arun, noting her emotional state and actions.
- Arun (person) [NEW] -> elt_f91e46621aa0
  aliases: -
  identification_keys: greenhouse worker; asks about Elias; holds tin cup; accompanies Mira to chapel
  update_instruction: Add Arun as Mira's companion who works in the greenhouse, questions the letter about Elias, and accompanies h

In [ ]:
#reject with feedback
elements_plan = propose_elements_update(
    diff_path="example_story_change.diff",
    elements_md_path="story/elements.md",
    reviewer_feedback="Do not create separate generic items unless they are clearly story-relevant. Keep identification keys shorter.",
)
preview_elements_proposal(elements_plan)

In [64]:
#approve
apply_elements_proposal(
    elements_plan,
    elements_md_path="story/elements.md",
    elements_dir="story/elements",
    approved=True,
)

Applied elements proposal to story/elements.md


In [65]:
## create elements manifest for next agent runs to update world model
run_path = apply_run_manifest(
    plan=elements_plan,
    runs_dir="story/runs",
    approved=True,
)

Applied run manifest to story/runs/2026-03-23T18-58-26.md


## sub agents

In [67]:
ELEMENT_PAGE_UPDATER_SYSTEM_PROMPT = """
You are the Element Page Updater.

Your job is to update exactly one world-model element page.

You receive:
1. the manuscript git diff
2. the approved run target for one UUID
3. the current parsed element object
4. the current raw markdown for that uuid.md file

Your task is to propose changes only for this one element page.

This page is not a flat extraction log.
It is a world-model dossier for this element.

Your goal is to make the element page:
- deeper
- more interpretive
- more reader-friendly
- more element-centered
- still faithful to the manuscript diff

Rules:
- You are not a critic. Do not judge the story as wrong, weird, implausible, or incomplete.
- Stay local to this one element. Do not update other elements.
- Use the diff as the source of truth for what changed.
- Preserve stable useful information unless the diff clearly adds, clarifies, or revises it.
- Do not invent unsupported facts.
- Prefer high-level chronology over minute-by-minute logging unless exact timing is clearly important.

Section guidance:

1. Core Understanding
Write 2-4 sentences that explain who or what this element is in the world, what role it plays, and what deeper tension or significance the recent diff sharpens.
This should not read like a scene recap.

2. Stable Profile
Add durable facts:
- roles
- relationships
- possessions
- recurring associations
- stable traits
Do not fill this section with one-off scene actions.

3. Interpretation
Add short bullets explaining what the recent diff suggests this means for the element.
This is where deeper carving belongs:
- emotional pressure
- secrecy
- motive
- role in unfolding mystery
- relationship implications
Keep it grounded.

4. Knowledge / Beliefs / Uncertainties
Track what this element appears to know, suspect, misunderstand, or still not know from its own perspective.

5. Element-Centered Chronology
Group chronology by chapter/date or era.
Use headings like:
- Before current narrative
- Chapter 7 — Late June 1998
- Chapter 8 — June 28, 1998
Within each heading, write 1-3 concise bullets from this element's perspective.
Do not log every minute unless the exact time is causally important.

6. Open Threads
Add unresolved questions, pressures, or unknowns that remain active for this element.

Output rules:
- Return only the structured proposal.
- Do not emit raw markdown patches.
- Do not emit full file content.
- Be concrete, concise, interpretive, and auditable.
"""

In [68]:
class RunTarget(BaseModel):
    uuid: str
    display_name: str
    kind: Literal["person", "place", "item", "animal", "concept", "group", "other"]
    file: str
    update_instruction: str


class ChronologyBlock(BaseModel):
    heading: str
    entries: list[str] = Field(default_factory=list)


class ParsedElementFile(BaseModel):
    uuid: str
    kind: str
    canonical_name: str
    aliases: list[str] = Field(default_factory=list)
    identification_keys: list[str] = Field(default_factory=list)

    core_understanding: str = ""
    stable_profile_lines: list[str] = Field(default_factory=list)
    interpretation_lines: list[str] = Field(default_factory=list)
    knowledge_lines: list[str] = Field(default_factory=list)
    chronology_blocks: list[ChronologyBlock] = Field(default_factory=list)
    open_threads_lines: list[str] = Field(default_factory=list)


class ChronologyBlockUpdate(BaseModel):
    heading: str = Field(
        description="High-level chronology bucket like 'Before current narrative' or 'Chapter 8 — June 28, 1998'"
    )
    entries: list[str] = Field(
        default_factory=list,
        description="1-3 concise bullets from the element's perspective"
    )


class ElementFileUpdateProposal(BaseModel):
    changed: bool = Field(description="Whether this uuid.md should change")
    rationale: str = Field(description="Why these changes are being proposed")

    core_understanding_replacement: Optional[str] = Field(
        default=None,
        description="Replacement text for Core Understanding, or null if unchanged"
    )

    stable_profile_to_add: list[str] = Field(default_factory=list)
    stable_profile_to_remove: list[str] = Field(default_factory=list)

    interpretation_to_add: list[str] = Field(default_factory=list)

    knowledge_to_add: list[str] = Field(default_factory=list)
    knowledge_to_remove: list[str] = Field(default_factory=list)

    chronology_blocks_to_add: list[ChronologyBlockUpdate] = Field(default_factory=list)

    open_threads_to_add: list[str] = Field(default_factory=list)
    open_threads_to_remove: list[str] = Field(default_factory=list)

    approval_message: str = Field(description="Short human-facing summary of the proposed change")

In [69]:
## parse run.md and parse/build uuid.md

def parse_run_manifest(run_path: str | Path) -> dict:
    text = read_text(run_path)
    lines = text.splitlines()

    run_id = None
    created_at = None
    targets: list[RunTarget] = []

    current = None
    in_targets = False

    for raw in lines:
        line = raw.rstrip()

        if line.startswith("- run_id:"):
            run_id = line.split(":", 1)[1].strip()
            continue

        if line.startswith("- created_at:"):
            created_at = line.split(":", 1)[1].strip()
            continue

        if line.strip() == "## Element Update Targets":
            in_targets = True
            continue

        if not in_targets:
            continue

        if line.startswith("- uuid:"):
            if current:
                targets.append(RunTarget(**current))
            current = {"uuid": line.split(":", 1)[1].strip()}
            continue

        if current and line.startswith("  "):
            key, value = line.strip().split(":", 1)
            current[key] = value.strip()

    if current:
        targets.append(RunTarget(**current))

    return {
        "run_id": run_id,
        "created_at": created_at,
        "targets": targets,
    }


def get_run_target(run_path: str | Path, target_uuid: str) -> RunTarget:
    manifest = parse_run_manifest(run_path)
    for target in manifest["targets"]:
        if target.uuid == target_uuid:
            return target
    raise ValueError(f"UUID not found in run manifest: {target_uuid}")

In [70]:
def _extract_section(text: str, start_heading: str, end_heading: Optional[str] = None) -> str:
    start = text.find(start_heading)
    if start == -1:
        return ""
    start += len(start_heading)

    if end_heading:
        end = text.find(end_heading, start)
        if end == -1:
            end = len(text)
    else:
        end = len(text)

    return text[start:end].strip()


def _parse_bullet_field(line: str, prefix: str) -> str:
    if line.startswith(prefix):
        return line[len(prefix):].strip()
    return ""


def _split_csv_field(value: str) -> list[str]:
    if not value or value == "-":
        return []
    return [x.strip() for x in value.split(",") if x.strip()]


def _split_semicolon_field(value: str) -> list[str]:
    if not value or value == "-":
        return []
    return [x.strip() for x in value.split(";") if x.strip()]


def _parse_bullet_lines(section_text: str) -> list[str]:
    items = []
    for raw in section_text.splitlines():
        line = raw.strip()
        if line.startswith("- "):
            value = line[2:].strip()
            if value and value != "TBD":
                items.append(value)
    return items


def _parse_chronology_blocks(section_text: str) -> list[ChronologyBlock]:
    section_text = section_text.strip()
    if not section_text or section_text == "- TBD":
        return []

    blocks: list[ChronologyBlock] = []
    current_heading = None
    current_entries: list[str] = []

    for raw in section_text.splitlines():
        line = raw.strip()

        if not line:
            continue

        if line.startswith("### "):
            if current_heading and current_entries:
                blocks.append(ChronologyBlock(heading=current_heading, entries=current_entries))
            current_heading = line[4:].strip()
            current_entries = []
            continue

        if line.startswith("- "):
            entry = line[2:].strip()
            if entry and entry != "TBD":
                if current_heading is None:
                    current_heading = "Ungrouped"
                current_entries.append(entry)

    if current_heading and current_entries:
        blocks.append(ChronologyBlock(heading=current_heading, entries=current_entries))

    return blocks


def parse_element_file(path: str | Path) -> ParsedElementFile:
    text = read_text(path)

    title_match = re.search(r"^#\s+(.+)$", text, flags=re.MULTILINE)
    canonical_name = title_match.group(1).strip() if title_match else ""

    identification_section = _extract_section(text, "## Identification", "## Core Understanding")
    if not identification_section:
        identification_section = _extract_section(text, "## Identification", "## Snapshot")

    uuid = ""
    kind = ""
    aliases = []
    identification_keys = []

    for raw in identification_section.splitlines():
        line = raw.strip()
        if line.startswith("- UUID:"):
            uuid = _parse_bullet_field(line, "- UUID:")
        elif line.startswith("- Type:"):
            kind = _parse_bullet_field(line, "- Type:")
        elif line.startswith("- Aliases:"):
            aliases = _split_csv_field(_parse_bullet_field(line, "- Aliases:"))
        elif line.startswith("- Identification keys:"):
            identification_keys = _split_semicolon_field(_parse_bullet_field(line, "- Identification keys:"))

    # New format
    core_understanding = _extract_section(text, "## Core Understanding", "## Stable Profile")
    stable_profile_section = _extract_section(text, "## Stable Profile", "## Interpretation")
    interpretation_section = _extract_section(text, "## Interpretation", "## Knowledge / Beliefs / Uncertainties")
    knowledge_section = _extract_section(text, "## Knowledge / Beliefs / Uncertainties", "## Element-Centered Chronology")
    chronology_section = _extract_section(text, "## Element-Centered Chronology", "## Open Threads")
    open_threads_section = _extract_section(text, "## Open Threads", None)

    # Backward compatibility with old format
    if not core_understanding:
        core_understanding = _extract_section(text, "## Snapshot", "## Attributes")

    if not stable_profile_section:
        stable_profile_section = _extract_section(text, "## Attributes", "## Timeline")

    old_timeline_section = _extract_section(text, "## Timeline", None)

    chronology_blocks = _parse_chronology_blocks(chronology_section)
    if not chronology_blocks:
        old_timeline_lines = _parse_bullet_lines(old_timeline_section)
        if old_timeline_lines:
            chronology_blocks = [ChronologyBlock(heading="Imported Timeline", entries=old_timeline_lines)]

    return ParsedElementFile(
        uuid=uuid,
        kind=kind,
        canonical_name=canonical_name,
        aliases=aliases,
        identification_keys=identification_keys,
        core_understanding=core_understanding.strip() if core_understanding.strip() != "TBD" else "",
        stable_profile_lines=_parse_bullet_lines(stable_profile_section),
        interpretation_lines=_parse_bullet_lines(interpretation_section),
        knowledge_lines=_parse_bullet_lines(knowledge_section),
        chronology_blocks=chronology_blocks,
        open_threads_lines=_parse_bullet_lines(open_threads_section),
    )

In [71]:
def build_element_markdown(obj: ParsedElementFile) -> str:
    aliases_str = ", ".join(obj.aliases) if obj.aliases else "-"
    keys_str = "; ".join(obj.identification_keys) if obj.identification_keys else "-"

    stable_profile = obj.stable_profile_lines if obj.stable_profile_lines else ["TBD"]
    interpretation = obj.interpretation_lines if obj.interpretation_lines else ["TBD"]
    knowledge = obj.knowledge_lines if obj.knowledge_lines else ["TBD"]
    open_threads = obj.open_threads_lines if obj.open_threads_lines else ["TBD"]

    lines = [
        f"# {obj.canonical_name}",
        "",
        "## Identification",
        f"- UUID: {obj.uuid}",
        f"- Type: {obj.kind}",
        f"- Canonical name: {obj.canonical_name}",
        f"- Aliases: {aliases_str}",
        f"- Identification keys: {keys_str}",
        "",
        "## Core Understanding",
        obj.core_understanding.strip() if obj.core_understanding.strip() else "TBD",
        "",
        "## Stable Profile",
    ]

    lines.extend([f"- {x}" for x in stable_profile])

    lines.extend([
        "",
        "## Interpretation",
    ])
    lines.extend([f"- {x}" for x in interpretation])

    lines.extend([
        "",
        "## Knowledge / Beliefs / Uncertainties",
    ])
    lines.extend([f"- {x}" for x in knowledge])

    lines.extend([
        "",
        "## Element-Centered Chronology",
    ])

    if obj.chronology_blocks:
        for block in obj.chronology_blocks:
            lines.append(f"### {block.heading}")
            for entry in block.entries:
                lines.append(f"- {entry}")
            lines.append("")
        if lines[-1] == "":
            lines.pop()
    else:
        lines.append("- TBD")

    lines.extend([
        "",
        "## Open Threads",
    ])
    lines.extend([f"- {x}" for x in open_threads])
    lines.append("")

    return "\n".join(lines)

In [72]:
element_page_model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=5000,
    max_retries=2,
    timeout=120,
).with_structured_output(ElementFileUpdateProposal)

In [73]:
def _normalize_line(x: str) -> str:
    return re.sub(r"\s+", " ", x.strip())


def _merge_section_lines(
    current_lines: list[str],
    to_add: list[str],
    to_remove: list[str],
) -> list[str]:
    result = [x for x in current_lines]

    remove_set = {_normalize_line(x) for x in to_remove if x.strip()}
    result = [x for x in result if _normalize_line(x) not in remove_set]

    existing_set = {_normalize_line(x) for x in result}
    for item in to_add:
        norm = _normalize_line(item)
        if norm and norm not in existing_set:
            result.append(item.strip())
            existing_set.add(norm)

    return result


def _merge_chronology_blocks(
    current_blocks: list[ChronologyBlock],
    new_blocks: list[ChronologyBlockUpdate],
) -> list[ChronologyBlock]:
    by_heading = {block.heading: ChronologyBlock(heading=block.heading, entries=list(block.entries)) for block in current_blocks}

    for block in new_blocks:
        heading = block.heading.strip()
        if not heading:
            continue

        if heading not in by_heading:
            by_heading[heading] = ChronologyBlock(heading=heading, entries=[])

        existing_norm = {_normalize_line(x) for x in by_heading[heading].entries}
        for entry in block.entries:
            norm = _normalize_line(entry)
            if norm and norm not in existing_norm:
                by_heading[heading].entries.append(entry.strip())
                existing_norm.add(norm)

    return list(by_heading.values())


def apply_update_proposal_to_object(
    current_obj: ParsedElementFile,
    proposal: ElementFileUpdateProposal,
) -> ParsedElementFile:
    if not proposal.changed:
        return current_obj.model_copy(deep=True)

    new_obj = current_obj.model_copy(deep=True)

    if proposal.core_understanding_replacement and proposal.core_understanding_replacement.strip():
        new_obj.core_understanding = proposal.core_understanding_replacement.strip()

    new_obj.stable_profile_lines = _merge_section_lines(
        current_lines=new_obj.stable_profile_lines,
        to_add=proposal.stable_profile_to_add,
        to_remove=proposal.stable_profile_to_remove,
    )

    new_obj.interpretation_lines = _merge_section_lines(
        current_lines=new_obj.interpretation_lines,
        to_add=proposal.interpretation_to_add,
        to_remove=[],
    )

    new_obj.knowledge_lines = _merge_section_lines(
        current_lines=new_obj.knowledge_lines,
        to_add=proposal.knowledge_to_add,
        to_remove=proposal.knowledge_to_remove,
    )

    new_obj.chronology_blocks = _merge_chronology_blocks(
        current_blocks=new_obj.chronology_blocks,
        new_blocks=proposal.chronology_blocks_to_add,
    )

    new_obj.open_threads_lines = _merge_section_lines(
        current_lines=new_obj.open_threads_lines,
        to_add=proposal.open_threads_to_add,
        to_remove=proposal.open_threads_to_remove,
    )

    return new_obj


def build_unified_diff(old_text: str, new_text: str, file_path: str) -> str:
    if old_text == new_text:
        return ""

    diff = unified_diff(
        old_text.splitlines(keepends=True),
        new_text.splitlines(keepends=True),
        fromfile=f"a/{file_path}",
        tofile=f"b/{file_path}",
    )
    return "".join(diff)

In [74]:
def propose_element_page_update(
    run_manifest_path: str | Path,
    target_uuid: str,
    diff_path: str | Path,
    reviewer_feedback: Optional[str] = None,
) -> dict:
    target = get_run_target(run_manifest_path, target_uuid)
    element_path = Path(target.file)

    current_raw_markdown = read_text(element_path)
    current_obj = parse_element_file(element_path)
    diff_text = read_text(diff_path)

    feedback_text = reviewer_feedback.strip() if reviewer_feedback else "None"

    user_prompt = f"""
Approved run target:
{target.model_dump_json(indent=2)}

Current parsed element object:
{current_obj.model_dump_json(indent=2)}

Current raw uuid.md markdown:
<current_markdown>
{current_raw_markdown}
</current_markdown>

Source manuscript diff:
<diff>
{diff_text}
</diff>

Reviewer feedback:
{feedback_text}

Task:
Propose updates for this one element page only.

Important:
- Make the page deeper and more interpretive, not just a recap.
- Keep chronology grouped by chapter/date or era.
- Use the element's perspective.
- Do not flood chronology with minute-by-minute bullets unless exact timing clearly matters.
- Stable Profile should hold durable truths, not one-off scene actions.
- Interpretation should explain what the diff means for this element.
"""

    proposal = element_page_model.invoke([
        {"role": "system", "content": ELEMENT_PAGE_UPDATER_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ])

    new_obj = apply_update_proposal_to_object(current_obj, proposal)
    new_raw_markdown = build_element_markdown(new_obj)
    patch_text = build_unified_diff(current_raw_markdown, new_raw_markdown, target.file)

    return {
        "target": target.model_dump(),
        "current_obj": current_obj.model_dump(),
        "proposal": proposal.model_dump(),
        "new_obj": new_obj.model_dump(),
        "old_content": current_raw_markdown,
        "new_content": new_raw_markdown,
        "patch": patch_text,
    }

In [75]:
def preview_element_page_update(result: dict) -> None:
    print("\n=== TARGET ===")
    print(f"{result['target']['display_name']} -> {result['target']['uuid']}")
    print(result["target"]["file"])

    print("\n=== RATIONALE ===")
    print(result["proposal"]["rationale"])

    print("\n=== APPROVAL MESSAGE ===")
    print(result["proposal"]["approval_message"])

    print("\n=== PATCH ===")
    print(result["patch"] or "[no change]")


def apply_element_page_update(
    result: dict,
    approved: bool = False,
) -> None:
    if not approved:
        raise PermissionError("Not applied. Pass approved=True only after review.")

    target_file = Path(result["target"]["file"])
    write_text(target_file, result["new_content"])
    print(f"Applied update to {target_file}")


def retry_element_page_update(
    run_manifest_path: str | Path,
    target_uuid: str,
    diff_path: str | Path,
    feedback: str,
) -> dict:
    return propose_element_page_update(
        run_manifest_path=run_manifest_path,
        target_uuid=target_uuid,
        diff_path=diff_path,
        reviewer_feedback=feedback,
    )

In [77]:
result = propose_element_page_update(
    run_manifest_path=run_path,
    target_uuid="elt_6ad4c9d565e1",
    diff_path="example_story_change.diff",
)

preview_element_page_update(result)


=== TARGET ===
Mira -> elt_6ad4c9d565e1
story/elements/elt_6ad4c9d565e1.md

=== RATIONALE ===
The diff adds extensive, time‑stamped details about Mira’s actions, possessions, and revelations in chapters 7 and 8. Updating the element page with a richer core understanding, durable profile facts, interpretive insights, explicit knowledge, a structured chronology, and open threads aligns the dossier with the manuscript while deepening the reader’s grasp of Mira’s emotional stakes and narrative role.

=== APPROVAL MESSAGE ===
Update Mira's element page to reflect new manuscript details, deepen profile, add interpretation, knowledge, chronology, and open threads.

=== PATCH ===
--- a/story/elements/elt_6ad4c9d565e1.md
+++ b/story/elements/elt_6ad4c9d565e1.md
@@ -8,20 +8,51 @@
 - Identification keys: protagonist; carries silver key; searches chapel; recalls Elias
 
 ## Core Understanding
-Mira is the central protagonist, a young woman haunted by her mother's letter and a mysterious silver ke

In [78]:
# Cell 11: approve

apply_element_page_update(
    result,
    approved=True,
)

Applied update to story/elements/elt_6ad4c9d565e1.md


In [ ]:
# Cell 10: reject + feedback

result = retry_element_page_update(
    run_manifest_path=run_path,
    target_uuid="elt_6ad4c9d565e1",
    diff_path="example_story_change.diff",
    feedback="Make Core Understanding calmer and less recap-like. Move one-off scene actions out of Stable Profile. Group chronology by chapter and date, not minute-by-minute.",
)

preview_element_page_update(result)